# 02 — Feature Engineering

Inputs: `data/case1_credit_approval.csv`  
Outputs: `data/features.csv`

| Feature | Source | 
|---|---|
| `AGE_YEARS` | `DAYS_BIRTH` / 365.25| 
| `EMPLOY_YEARS` | `DAYS_EMPLOYED` / 365.25 | 
| `CREDIT_TO_INCOME` | `AMT_CREDIT / AMT_INCOME_TOTAL` | 
| `ANNUITY_TO_INCOME` | `AMT_ANNUITY / AMT_INCOME_TOTAL` | 
| `DEBT_TO_INCOME` | `BUREAU_TOTAL_DEBT / AMT_INCOME_TOTAL` | 

## 0. Imports

In [11]:
import pandas as pd
import pathlib
import warnings

warnings.filterwarnings('ignore')

DATA_PATH   = '../data/case1_credit_approval.csv'
OUTPUT_PATH = '../data/features.csv'
RANDOM_STATE = 42

DROP_COLS = ['SK_ID_CURR', 'APPLICANT_SCENARIO']

## 1. Load data and drop 

In [12]:
df = pd.read_csv(DATA_PATH)
print(f'Raw shape: {df.shape}')
df = df.drop(columns=DROP_COLS)

Raw shape: (50000, 28)


## 2. N/a columns in `BUREAU_*` Vars

In [13]:
bureau_zero_cols = [
    'BUREAU_ACTIVE_LOANS',
    'BUREAU_CLOSED_LOANS',
    'BUREAU_TOTAL_DEBT',
    'BUREAU_TOTAL_CREDIT_LIMIT',
    'BUREAU_UTILIZATION',
    'BUREAU_DPD_30_COUNT',
    'BUREAU_DPD_60_COUNT',
    'BUREAU_DPD_90_COUNT',
    'BUREAU_INQUIRIES_6M',
]

df[bureau_zero_cols] = df[bureau_zero_cols].fillna(0)

print('Null vals after fill:')
print(df[bureau_zero_cols].isnull().sum())
print(f'\nThin-file rows (HAS_BUREAU_FILE = 0): {(df["HAS_BUREAU_FILE"] == 0).sum()}')

Null vals after fill:
BUREAU_ACTIVE_LOANS          0
BUREAU_CLOSED_LOANS          0
BUREAU_TOTAL_DEBT            0
BUREAU_TOTAL_CREDIT_LIMIT    0
BUREAU_UTILIZATION           0
BUREAU_DPD_30_COUNT          0
BUREAU_DPD_60_COUNT          0
BUREAU_DPD_90_COUNT          0
BUREAU_INQUIRIES_6M          0
dtype: int64

Thin-file rows (HAS_BUREAU_FILE = 0): 8752


`HAS_BUREAU_FILE` is kept as an explicit flag and the nine structural bureau columns are filled with 0. Because the flag is retained, no information is lost as the model can still separate "no credit file at all" (flag 0, everything 0) from "has a file but zero activity" (flag 1, a real 0). We deliberately leave `BUREAU_MONTHS_SINCE_LAST_DELINQUENCY` unfilled. Its NA is itself meaningful.

Thin-file decisions can only rest on demographic variables, declared income and limited bureau data, with genuine repayment behaviour observable only after a 6–12 month seasoning period. Predictive power for this group is therefore weaker.

In practice, lenders will segment the book into two populations, those with and without a credit history, and model them separately, because the risk drivers differ. Using `HAS_BUREAU_FILE` is therefore a modelling decision allowing us to use a single pooled model to serve both populations. Its key shortcoming is that it shares one set of coefficients across both groups, as the flag shifts the baseline risk for thin-file applicants, but the model cannot learn genuinely different feature relationships for them.

**Objective 3: Create features such as credit-to-income, annuity-to-income, age, employment duration, and debt-to-income**

**Objective 4: Explain the business intuition behind each feature**

## 3. `AGE_YEARS`

`DAYS_BIRTH` / 365.25

In [14]:
df['AGE_YEARS'] = (-df['DAYS_BIRTH']) / 365.25

print(f'AGE_YEARS — min: {df["AGE_YEARS"].min():.1f}, max: {df["AGE_YEARS"].max():.1f}, mean: {df["AGE_YEARS"].mean():.1f}')
df = df.drop(columns=['DAYS_BIRTH'])

AGE_YEARS — min: 21.0, max: 70.9, mean: 41.0


Converted to years for interpretability in model output. Age is a consistent predictor in consumer credit scoring: younger borrowers typically have shorter credit histories, higher income volatility, and less experience managing debt.

## 4. `EMPLOY_YEARS`

`DAYS_EMPLOYED` / 365.25

In [15]:
df['EMPLOY_YEARS'] = (-df['DAYS_EMPLOYED']) / 365.25

print(f'EMPLOY_YEARS — min: {df["EMPLOY_YEARS"].min():.1f}, max: {df["EMPLOY_YEARS"].max():.1f}, mean: {df["EMPLOY_YEARS"].mean():.1f}')
print(f'Null values: {df["EMPLOY_YEARS"].isnull().sum()}')
df = df.drop(columns=['DAYS_EMPLOYED'])

EMPLOY_YEARS — min: 0.0, max: 44.4, mean: 18.0
Null values: 0


Employment income is a proxy for income stability, A borrower in stable employment for several years presents materially lower default risk than a recent joiner, as longer tenure reduces the probability that income disappears before loan repayment is complete.

## 5. `CREDIT_TO_INCOME`

`AMT_CREDIT` / `AMT_INCOME_TOTAL`

In [16]:
df['CREDIT_TO_INCOME'] = df['AMT_CREDIT'] / df['AMT_INCOME_TOTAL']

print(f'CREDIT_TO_INCOME — min: {df["CREDIT_TO_INCOME"].min():.2f}, max: {df["CREDIT_TO_INCOME"].max():.2f}, median: {df["CREDIT_TO_INCOME"].median():.2f}')
print(f'Null values: {df["CREDIT_TO_INCOME"].isnull().sum()}')

CREDIT_TO_INCOME — min: 0.25, max: 71.47, median: 2.04
Null values: 523


A stock measure of affordability capturing how large the requested loan is relative to annual income. Empirically a primary driver of PD for both mortgage and consumer loans. Relationship between CTI and default risk is convex, at high ratios the borrower has exhausted their capacity to absorb income shocks, causing risk to escalate in a non-linear fashion.

## 6. `ANNUITY_TO_INCOME`

`AMT_ANNUITY` / `AMT_INCOME_TOTAL`

In [17]:
df['ANNUITY_TO_INCOME'] = df['AMT_ANNUITY'] / df['AMT_INCOME_TOTAL']

print(f'ANNUITY_TO_INCOME — min: {df["ANNUITY_TO_INCOME"].min():.4f}, max: {df["ANNUITY_TO_INCOME"].max():.4f}, median: {df["ANNUITY_TO_INCOME"].median():.4f}')
print(f'Null values: {df["ANNUITY_TO_INCOME"].isnull().sum()}')

ANNUITY_TO_INCOME — min: 0.0058, max: 2.1767, median: 0.0688
Null values: 523


Measures monthly flow of repayments relative to income, or what fraction of take-home pay disappears every month to service the loan. Consider 2 identical applicants with the same income and loan (so same CTI), but with a different monthly payment (perhaps person A has a short-term high interest loan and person B has a long-term low interest loan), A would have higher ATI, as ATI is a flow measure, while CTI is a stock measure.

Empirically, the level of indebtedness measured by the debt-service-to-income ratio (which ATI is an approximation of) is one of the main drivers of probability of default for consumer loans. It captures vulnerability to variable rate loans, encompassing repayment shock risk that CTI misses.

## 7. `DEBT_TO_INCOME`

`BUREAU_TOTAL_DEBT` / `AMT_INCOME_TOTAL`

In [18]:
df['DEBT_TO_INCOME'] = df['BUREAU_TOTAL_DEBT'] / df['AMT_INCOME_TOTAL']

print(f'DEBT_TO_INCOME — min: {df["DEBT_TO_INCOME"].min():.2f}, max: {df["DEBT_TO_INCOME"].max():.2f}, median: {df["DEBT_TO_INCOME"].median():.2f}')
print(f'Zero values (no bureau debt): {(df["DEBT_TO_INCOME"] == 0).sum()}')
print(f'Null values: {df["DEBT_TO_INCOME"].isnull().sum()}')

DEBT_TO_INCOME — min: 0.00, max: 88.26, median: 0.22
Zero values (no bureau debt): 19640
Null values: 523


A stock measure of total external obligations across all lenders, not just current loan. An applicant may appear affordable based off of this loan alone, while carrying significant obligations elsewhere. 

## 8. Recoding `BUREAU_MONTHS_SINCE_LAST_DELINQUENCY`

I tried a few approaches here, this seemed to make the most sense. The issue was what to do with `BUREAU_MONTHS_SINCE_LAST_DELINQUENCY`, it contains useful information but it was unclear how to deal with the N/a entries. Imputing with 0 of course was nonsensical, imputing with the largest value/999 made some sense but of course was not true. Dropping the column risked losing valuable data. The binning approach seemed to make the most sense, converting this mix of numeric + missing data into categories. 

In [19]:
df['MONTHS_SINCE_DELINQ_BINNED'] = pd.cut(
    df['BUREAU_MONTHS_SINCE_LAST_DELINQUENCY'],
    bins=[-0.1, 12, 24, 60, float('inf')],
    labels=['recent_0_12m', 'moderate_13_24m', 'older_25_60m', 'distant_60m_plus']
).astype(str)

df['MONTHS_SINCE_DELINQ_BINNED'] = df['MONTHS_SINCE_DELINQ_BINNED'].fillna('never_delinquent')

df = df.drop(columns=['BUREAU_MONTHS_SINCE_LAST_DELINQUENCY'])

print(df['MONTHS_SINCE_DELINQ_BINNED'].value_counts())
print(f'\nDefault rate by bin:')
print(df.groupby('MONTHS_SINCE_DELINQ_BINNED')['TARGET']
      .agg(default_rate='mean', count='count').round(4))

MONTHS_SINCE_DELINQ_BINNED
never_delinquent    34144
older_25_60m         5678
recent_0_12m         4686
distant_60m_plus     3569
moderate_13_24m      1923
Name: count, dtype: int64

Default rate by bin:
                            default_rate  count
MONTHS_SINCE_DELINQ_BINNED                     
distant_60m_plus                  0.1199   3569
moderate_13_24m                   0.1258   1923
never_delinquent                  0.0578  34144
older_25_60m                      0.1176   5678
recent_0_12m                      0.1571   4686


## 9. Save

In [20]:
pathlib.Path('../data').mkdir(exist_ok=True)
df.to_csv(OUTPUT_PATH, index=False)

print(f'Saved to {OUTPUT_PATH}')
print(f'Final shape: {df.shape}')

Saved to ../data/features.csv
Final shape: (50000, 29)
